In [3]:
import os 
from patient import Patient
import pandas as pd
import numpy as np
from sklearn.feature_selection import f_classif
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, StratifiedGroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import json
from sklearn.neighbors import KNeighborsClassifier
from feature_engine.selection import MRMR

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from concurrent.futures import ProcessPoolExecutor
import functools



rng = np.random.default_rng(seed=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
#relevant methods

def calculate_metrics(y_true, y_pred):
    """Calcualtes accuracy, sensitivity and specificity by macro-averaging over all classes."""

    cm = confusion_matrix(y_true, y_pred)

    sensitivities = []
    specificities = []

    for i in range(len(cm)):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        tn = cm.sum() - (tp + fp + fn)

        sensitivity = (tp / (tp + fn) if (tp + fn) > 0 else 0)
        specificity = (tn / (tn + fp) if (tn + fp) > 0 else 0)

        sensitivities.append(sensitivity)
        specificities.append(specificity)

    # Overall accuracy
    accuracy = cm.diagonal().sum() / cm.sum()

    # Macro-average across all classes
    sensitivity = sum(sensitivities) / len(sensitivities)
    specificity = sum(specificities) / len(specificities)

    return accuracy, sensitivity, specificity, cm

def clean_dataset(dataset):
    """Clean dataset and return dataset with features for ECG+SpO2, SpO2 only or ECG only."""   

    # nans are a problem for F-test, thus need to be found
    print("NaNs in dataset:")
    print(dataset.isna().sum()[dataset.isna().sum() > 0])

    # nan are replaced with median of column
    dataset = dataset.fillna(dataset.median(numeric_only=True))

    # create dataframes with only SpO2 or ECG features (and the age, subject, etc)
    spo2_columns = ["Smini","Smean","Svari","CorreC","NumZC","Slope","AbsSlope","Bias","Dmean",
                    "ApEnt","LZCom","CTM25","CTM50","CTM75","CTM100","odi2","odi3","odi4","ODI21",
                    "ODI23","ODI25","ODI31","ODI33","ODI35","ODI41","ODI43","ODI45","ODI51","ODI53",
                    "ODI55","ODIS2","ODIS3","ODIS4","ODIS5","tsa70","tsa80","tsa85","tsa90","tsa95"]

    dataset_spo2 = dataset[["Apnea", "Subject", "Age", "Sex", "AHI"] + spo2_columns]
    dataset_ecg = dataset.drop(spo2_columns, axis=1)

    # apply the corrleation function with LOPO F-scoring to reduce the amount of features 
    dataset_reduced_SpO2_ECG = find_reduced_feature_set(dataset, correlation_threshold=0.75)
    dataset_reduced_SpO2 = find_reduced_feature_set(dataset_spo2, correlation_threshold=0.75)
    dataset_reduced_ECG  = find_reduced_feature_set(dataset_ecg, correlation_threshold=0.75)

    return dataset_reduced_SpO2_ECG, dataset_reduced_SpO2, dataset_reduced_ECG


def clean_dataset2(dataset):
    """Clean dataset and return dataset with features for ECG+SpO2, SpO2 only or ECG only."""   

    # nans are a problem for F-test, thus need to be found
    print("NaNs in dataset:")
    print(dataset.isna().sum()[dataset.isna().sum() > 0])

    # nan are replaced with median of column
    dataset = dataset.fillna(dataset.median(numeric_only=True))

    # create dataframes with only SpO2 or ECG features (and the age, subject, etc)
    spo2_columns = ["Smini","Smean","Svari","CorreC","NumZC","Slope","AbsSlope","Bias","Dmean",
                    "ApEnt","LZCom","CTM25","CTM50","CTM75","CTM100","odi2","odi3","odi4","ODI21",
                    "ODI23","ODI25","ODI31","ODI33","ODI35","ODI41","ODI43","ODI45","ODI51","ODI53",
                    "ODI55","ODIS2","ODIS3","ODIS4","ODIS5","tsa70","tsa80","tsa85","tsa90","tsa95"]

    dataset_spo2 = dataset[["Apnea", "Subject", "Age", "Sex", "AHI"] + spo2_columns]
    dataset_ecg = dataset.drop(spo2_columns, axis=1)

    return dataset, dataset_spo2, dataset_ecg

def find_reduced_feature_set(df, correlation_threshold=0.75):
    
    """
    Find reduced feature set be calculating F-test leaving one subject out. For features with high F-score above threshold,
    pairs are formed, and the one with lowest F-score is kept. 
    
    Following the implementation idea by Deviaene, Margot et al. (2019). “Automatic Screening of Sleep Apnea Patients Based on the SpO2 Signal”.
    In: IEEE Journal of Biomedical and Health Informatics 23.2, pp. 607–617. DOI: https://doi.org/10.1109/JBHI.2018.2817368
    """
    
    feature_cols = [col for col in df.columns if col not in ["Apnea", "Subject", "Age", "Sex", "AHI"]]
    scaler = StandardScaler()
    df_scaled = df.copy()
    df_scaled[feature_cols] = scaler.fit_transform(df[feature_cols])

    X = df_scaled.drop(columns=["Apnea", "Subject", "Age", "Sex", "AHI"])
    y = df["Apnea"]
    subject_ids = df["Subject"]
    
    print("Computing LOPO F-scores... (this may take a while)")
    unique_patients = subject_ids.unique()
    all_lopo_scores = []

    for patient in unique_patients:
        # Leave one patient out
        X_train = X[subject_ids != patient] # this creates a boolean mask and keeps rows where conditons is true 
        y_train = y[subject_ids != patient]
        
        # f_classif computes the F-test for all features at once
        f_scores, _ = f_classif(X_train, y_train)
        all_lopo_scores.append(f_scores)

    # Take the median of F-scores across all LOPO iterations
    median_f_scores = np.nanmedian(np.array(all_lopo_scores), axis=0)

    # Handle cases where a feature was NaN in EVERY single fold
    median_f_scores = np.nan_to_num(median_f_scores, nan=0.0)
    
    f_score_map = dict(zip(X.columns, median_f_scores))
  
    corr_matrix = X.corr().abs()
    
    # upper triangle of the correlation matrix 
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    
    to_drop = set()
    pairs = []
    for col in upper.columns:
        for row in upper.index:
            if upper.loc[row, col] > correlation_threshold:
                pairs.append((row, col, upper.loc[row, col]))
    
    pairs.sort(key=lambda x: x[2], reverse=True)

    for feat_a, feat_b, corr_val in pairs:
    
        if feat_a in to_drop or feat_b in to_drop:
            continue
        
        # Compare F-scores: delete the one with the lowest score
        if f_score_map[feat_a] < f_score_map[feat_b]:
            to_drop.add(feat_a)
        else:
            to_drop.add(feat_b)

    print(f"Dropped {len(to_drop)} features. Retained {X.shape[1] - len(to_drop)} features.")
    
    # Return the dataframe with only the retained features
    retained_features = [col for col in X.columns if col not in to_drop]

    return df[["Apnea", "Subject", "Age", "Sex", "AHI"] + retained_features ]


def apply_MRMR_framework(dataset):
    """Maximum Relevance Minimum Redundancy framework
    https://feature-engine.trainindata.com/en/1.8.x/user_guide/selection/MRMR.html """

    # scaling necessary?
    # averaging all samples for one patient?

    X, y = dataset.drop(["Apnea","Subject","Age","Sex","AHI"], axis=1), dataset["Apnea"]

    sel = MRMR(method="MID", regression=False)
    sel.fit(X, y)

    # only keep the features which were not removed
    Xtr = sel.transform(X)

    new_df = pd.concat([dataset[["Apnea","Subject","Age","Sex","AHI"]], Xtr], axis=1)

    print(f"Dropped {len(sel.features_to_drop_)} features from {len(new_df.columns)-5}")

    return new_df

def apply_MRMR_framework_percent(dataset, keep_percent):
    """Maximum Relevance Minimum Redundancy framework
    https://feature-engine.trainindata.com/en/1.8.x/user_guide/selection/MRMR.html 
    
    keep percent - percentage of features to keep between 0-1
    
    """

    # scaling necessary?
    # averaging all samples for one patient?

    X, y = dataset.drop(["Apnea","Subject","Age","Sex","AHI"], axis=1), dataset["Apnea"]

    number_of_features = int(X.shape[1] * keep_percent)

    sel = MRMR(method="MID", max_features=number_of_features, regression=False)
    sel.fit(X, y)

    # only keep the features which were not removed
    Xtr = sel.transform(X)

    new_df = pd.concat([dataset[["Apnea","Subject","Age","Sex","AHI"]], Xtr], axis=1)

    print(f"Dropped {len(sel.features_to_drop_)} features from {len(new_df.columns)-5}")

    return new_df

def get_features_in_jsonl(filepath):
    #filepath = 'log_results_RF_A50_N50_H00.jsonl'

    with open(filepath, 'r') as f:
        lines = f.readlines()
        last_line = lines[-1] if lines else None

    if last_line:
        data = json.loads(last_line)


    return data["feature_importances"]


In [5]:
def grid_search_RF(dataset, identifier, classes, filepath):
    "Can be used for both binary and multiclass prediction."

    X_c, y = dataset.drop(["Apnea","Subject","Age","Sex","AHI"], axis=1), dataset["Apnea"]
    groups = dataset["Subject"].values

    # converto numpy array
    X = X_c.values
    y = y.values

    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    train_index, val_index = next(sgkf.split(X, y, groups)) # (take on split of a 5 cross-fold = 80%)

    X_train_full = X[train_index]
    X_val_final = X[val_index]
    y_train_full = y[train_index]
    y_val_final = y[val_index]
    groups_train_full = groups[train_index]

    max_accuracy = 0.0
    best_params = None

    for estimator in [50]:
        for max_depth in [5,10,15,20, None]:
            for min_samples_split in [2,4,8]:
                for max_features in ["sqrt", "log2", None]:
                    for criterion in ["gini", "entropy", "log_loss"]:

                        kf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42) # better than KFold, as it presevers proprotion of classes in each fold
                        model = RandomForestClassifier(
                            n_estimators=estimator,
                            max_depth=max_depth,
                            min_samples_split=min_samples_split,
                            max_features=max_features,
                            random_state=42,
                            criterion=criterion,
                            n_jobs=-1)

                        accuracies = []

                        for fold, (train_index, test_index) in enumerate(kf.split(X_train_full, y_train_full, groups_train_full), 1):
                            
                            X_train, X_test = X_train_full[train_index], X_train_full[test_index]
                            y_train, y_test = y_train_full[train_index], y_train_full[test_index]
                            
                            # training 
                            model.fit(X_train, y_train)
                            predictions = model.predict(X_test)

                            acc, _, _, _ = calculate_metrics(y_test, predictions)
                            accuracies.append(acc)


                        current_mean_acc = float(np.mean(accuracies))
                
                        if current_mean_acc > max_accuracy:
                            max_accuracy = current_mean_acc
                            best_params = {
                                "estimator": estimator, 
                                "max_depth": max_depth, 
                                "min_samples_split": min_samples_split, 
                                "max_features": max_features,
                                "criterion": criterion
                            }


    # fit best model with all data and test with validation set 
    
    final_model = RandomForestClassifier(
        n_estimators=best_params["estimator"], 
        max_depth=best_params["max_depth"], 
        min_samples_split=best_params["min_samples_split"], 
        max_features=best_params["max_features"], 
        random_state=42,
        criterion=best_params["criterion"],
        n_jobs=-1
    )
    final_model.fit(X_train_full, y_train_full)
    y_pred = final_model.predict(X_val_final)

    acc, sen, spe, cm = calculate_metrics(y_val_final, y_pred)

    importances = final_model.feature_importances_
    feature_names = X_c.columns

    feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
    feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False).reset_index(drop=True)
    features = feature_importance_df["Feature"].tolist()

    results = {
        "filepath": filepath,
        "algorithm": "RF",
        "identifier": identifier,
        "classes": classes,
        "test_acc": float(max_accuracy),
        "val_acc": float(acc),
        "val_sen": float(sen),
        "val_spe": float(spe),
        "params": best_params,
        "feature_importances": features,
        "cm" : cm.tolist()
    }

    with open(filepath, "a") as f:
        json.dump(results, f)
        f.write('\n')

    print("Grid Search completed.")

In [ ]:
def grid_search_kNN(dataset, identifier, classes, filepath):

    X, y = dataset.drop(["Apnea","Subject","Age","Sex","AHI"], axis=1), dataset["Apnea"]

    X = X.values
    y = y.values

    groups = dataset["Subject"].values

    kf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)
    scaler = StandardScaler()

    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    train_index, val_index = next(sgkf.split(X, y, groups)) # (take on split of a 5 cross-fold = 80%)
    
    X_train_full = X[train_index]
    X_val_final = X[val_index]
    y_train_full = y[train_index]
    y_val_final = y[val_index]
    groups_train_full = groups[train_index]
    
    max_accuracy = 0.0
    best_k = 0

    for k in [4,6,8,10,12,14,16,18,20,22,24,30,40,50]:

        accuracies = []
        model = KNeighborsClassifier(n_neighbors=k)

        for fold, (train_index, test_index) in enumerate(kf.split(X_train_full, y_train_full, groups_train_full), 1):
                                    
            X_train, X_test = X_train_full[train_index], X_train_full[test_index]
            y_train, y_test = y_train_full[train_index], y_train_full[test_index]
            
            # scaling: important to do that for X and Y seperately and for each fold, as the scaling depends on group charecteristics!! (mean)
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)
    
            model.fit(X_train_scaled, y_train)
            y_pred = model.predict(X_test_scaled)

            acc, _, _, cm = calculate_metrics(y_test, y_pred)
            accuracies.append(acc)

        if np.mean(accuracies) >= max_accuracy:

            max_accuracy = np.mean(accuracies)
            best_k = k


    # Training of the best model on full dataset and validation on held out set  

    scaler_val = StandardScaler()
    X_train_full_scaled = scaler_val.fit_transform(X_train_full)
    X_val_final_scaled = scaler_val.transform(X_val_final)

    model_val = KNeighborsClassifier(n_neighbors=best_k)
    model_val.fit(X_train_full_scaled, y_train_full)
    y_pred = model_val.predict(X_val_final_scaled)

    acc, sen, spe, cm = calculate_metrics(y_val_final, y_pred)

    results = {
        "filepath": filepath,
        "identifier": identifier,
        "classes": classes,
        "model": "kNN",
        "k": best_k,
        "test_acc": float(max_accuracy),
        "val_acc": float(acc),
        "val_sen": float(sen),
        "val_spe": float(spe),
        "cm": cm.tolist()
    }

    with open(filepath, "a") as f:
        json.dump(results, f)
        f.write('\n')
    

In [7]:
class ApneaNet(nn.Module):
    def __init__(self, input_dim, hidden_layer1:int, hidden_layer2:int, output:int):
        super(ApneaNet, self).__init__()
        self.layer1 = nn.Linear(input_dim, hidden_layer1)
        self.layer2 = nn.Linear(hidden_layer1, hidden_layer2)
        self.output_layer = nn.Linear(hidden_layer2, output) 
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        x = self.output_layer(x) # Return raw logits
        return x

In [ ]:
def grid_search_binary_MLP(dataset, identifier, classes, filepath):
    X_df = dataset.drop(["Apnea","Subject","Age","Sex","AHI"], axis=1)
    y_ser = dataset["Apnea"]
    
    X = X_df.values
    groups = dataset["Subject"].values

    # Encode classes as integers
    le = LabelEncoder()
    y = le.fit_transform(y_ser)

    # 20% held out for final validation
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    train_index, val_index = next(sgkf.split(X, y, groups))

    X_train_full, X_val_final = X[train_index], X[val_index]
    y_train_full, y_val_final = y[train_index], y[val_index]
    groups_train_full = groups[train_index]

    input_size = int(X.shape[1])
    hidden1 = int((2/3*input_size)+1)
    layer_sizes = [
        (hidden1, int(hidden1/2) if int(hidden1/2) > 1 else 2), 
        (int(hidden1*1.5), hidden1), 
        (int(hidden1*2.0), int(hidden1*1.5)), 
        (int(hidden1*3.0), int(hidden1*1.5)), 
        (int(input_size*3), int(input_size*0.5) if int(input_size*0.5) > 1 else 2 )
    ]

    kf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)

    max_accuracy = 0.0
    best_params = None

    for (hiddenlayer1, hiddenlayer2) in layer_sizes:
        print(f"Progress: {(hiddenlayer1, hiddenlayer2)}")
        accuracies = []

        
        for fold, (train_idx, test_idx) in enumerate(kf.split(X_train_full, y_train_full, groups_train_full), 1):

            # setup fold data
            X_train_fold, X_test_fold = X_train_full[train_idx], X_train_full[test_idx]
            y_train_fold, y_test_fold = y_train_full[train_idx], y_train_full[test_idx]

            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train_fold)
            X_test_scaled = scaler.transform(X_test_fold)

            # split Train fold into train and early-stopping set
            X_tr, X_es, y_tr, y_es = train_test_split(
                X_train_scaled, y_train_fold, test_size=0.2, stratify=y_train_fold, random_state=42
            )

            X_tr_t = torch.FloatTensor(X_tr).to(device)
            X_es_t = torch.FloatTensor(X_es).to(device)
            X_test_t = torch.FloatTensor(X_test_scaled).to(device)
            y_tr_t = torch.FloatTensor(y_tr).view(-1,1).to(device)
            y_es_t = torch.FloatTensor(y_es).view(-1,1).to(device)
            y_test_t = torch.FloatTensor(y_test_fold).view(-1,1).to(device)

            train_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=32, shuffle=True)

            model = ApneaNet(input_size, hiddenlayer1, hiddenlayer2, 1).to(device)
            criterion = nn.BCEWithLogitsLoss()
            optimizer = optim.Adam(model.parameters(), lr=0.001)

            # early stopping
            patience = 15
            best_fold_loss = float('inf')
            epochs_no_improve = 0
            best_model_state = None

            for epoch in range(200):
                model.train()
                for batch_X, batch_y in train_loader:
                    optimizer.zero_grad()
                    loss = criterion(model(batch_X), batch_y)
                    loss.backward()
                    optimizer.step()

                model.eval()
                with torch.no_grad():
                    curr_val_loss = criterion(model(X_es_t), y_es_t).item()
                    if curr_val_loss < best_fold_loss:
                        best_fold_loss = curr_val_loss
                        epochs_no_improve = 0
                        best_model_state = model.state_dict().copy()
                    else:
                        epochs_no_improve += 1
                
                if epochs_no_improve >= patience:
                    break

            # load best weights for the fold before evaluating on test set
            model.load_state_dict(best_model_state)
            model.eval()
            with torch.no_grad():
                test_outputs = model(X_test_t)
                test_preds = (test_outputs > 0).float()
                test_acc = (test_preds == y_test_t).sum().item() / len(y_test_t)
                accuracies.append(test_acc)

        current_mean_accuracy = np.mean(accuracies)
        if current_mean_accuracy > max_accuracy:
            max_accuracy = current_mean_accuracy
            best_params = (hiddenlayer1, hiddenlayer2)

    # train best model on all samples 
    scaler_final = StandardScaler()
    X_train_full_scaled = scaler_final.fit_transform(X_train_full)
    X_val_final_scaled = scaler_final.transform(X_val_final)

    X_train_full_t = torch.FloatTensor(X_train_full_scaled).to(device)
    X_val_final_t = torch.FloatTensor(X_val_final_scaled).to(device)
    y_train_full_t = torch.FloatTensor(y_train_full).view(-1,1).to(device)

    train_loader_final = DataLoader(TensorDataset(X_train_full_t, y_train_full_t), batch_size=32, shuffle=True)

    final_model = ApneaNet(input_size, best_params[0], best_params[1], 1).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(final_model.parameters(), lr=0.001)

    for epoch in range(100):
        final_model.train()
        for batch_X, batch_y in train_loader_final:
            optimizer.zero_grad()
            loss = criterion(final_model(batch_X), batch_y)
            loss.backward()
            optimizer.step()

    # final validation
    final_model.eval()
    with torch.no_grad():
        val_outputs = final_model(X_val_final_t)
        val_preds = (val_outputs > 0).float() # because logits 0.5 is 0
        
        final_val_acc = (val_preds == torch.FloatTensor(y_val_final).view(-1,1).to(device)).sum().item() / len(y_val_final)
        
        y_true_np = y_val_final
        y_pred_np = val_preds.cpu().numpy().flatten()
        
        tp = np.sum((y_pred_np == 1) & (y_true_np == 1))
        tn = np.sum((y_pred_np == 0) & (y_true_np == 0))
        fp = np.sum((y_pred_np == 1) & (y_true_np == 0))
        fn = np.sum((y_pred_np == 0) & (y_true_np == 1))
        
        cm = [[int(tn), int(fp)], [int(fn), int(tp)]]
        val_sensitivity = float(tp / (tp + fn) if (tp + fn) > 0 else 0)
        val_specificity = float(tn / (tn + fp) if (tn + fp) > 0 else 0)

    results = {
        "filepath": filepath,
        "identifier": identifier,
        "model": "MLP binary",
        "classes": classes,
        "params": best_params,
        "test_acc": float(max_accuracy),
        "val_acc": float(final_val_acc),
        "val_sen": val_sensitivity,
        "val_spe": val_specificity,
        "cm": cm
    }

    with open(filepath, "a") as f:
        json.dump(results, f)
        f.write('\n')

In [ ]:
def grid_search_MLP_three_classes(dataset, identifier, classes, filepath):
   
    X, y = dataset.drop(["Apnea","Subject","Age","Sex","AHI"], axis=1), dataset["Apnea"]
    X = X.values
    y = y.values
    groups = dataset["Subject"].values

    max_accuracy = 0.0
    le = LabelEncoder()
    y = le.fit_transform(y)


    # 20% validation set 
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    train_index, val_index = next(sgkf.split(X, y, groups))

    X_train_full, X_val_final = X[train_index], X[val_index]
    y_train_full, y_val_final = y[train_index], y[val_index]
    groups_train_full = groups[train_index]

    input_size = int(X.shape[1])
    hidden1 = int((2/3*input_size)+1)
    layer_sizes = [(hidden1, int(hidden1/2) if int(hidden1/2) > 1 else 3), (int(hidden1*1.5), hidden1), (int(hidden1*2.0), int(hidden1*1.5)), (int(hidden1*3.0), int(hidden1*1.5)), (int(input_size*3), int(input_size*0.5) if int(input_size*0.5) > 1 else 3 )]

    kf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)

    for (hiddenlayer1, hiddenlayer2) in layer_sizes:
        accuracies = []
        
        for fold, (train_index, test_index) in enumerate(kf.split(X_train_full, y_train_full, groups_train_full), 1):
            X_train, X_test = X_train_full[train_index], X_train_full[test_index]
            y_train, y_test = y_train_full[train_index], y_train_full[test_index]

            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)

            # 20% split for early stopping 
            X_tr, X_es, y_tr, y_es = train_test_split(X_train_scaled, y_train, test_size=0.2, stratify=y_train, random_state=42)
            
            X_tr_t = torch.FloatTensor(X_tr).to(device)
            X_es_t = torch.FloatTensor(X_es).to(device)
            X_test_t = torch.FloatTensor(X_test_scaled).to(device)
            y_tr_t = torch.LongTensor(y_tr).to(device)
            y_es_t = torch.LongTensor(y_es).to(device)
            y_test_t = torch.LongTensor(y_test).to(device)

            train_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=32, shuffle=True)

            model = ApneaNet(input_size, hidden_layer1=hiddenlayer1, hidden_layer2=hiddenlayer2, output=3).to(device)
            criterion = nn.CrossEntropyLoss() 
            optimizer = optim.Adam(model.parameters(), lr=0.001)

            # Early Stopping Variables
            patience, best_fold_loss, epochs_no_improve, best_model_state = 15, float('inf'), 0, None

            for epoch in range(200):
                model.train() 
                for batch_X, batch_y in train_loader:
                    optimizer.zero_grad()
                    loss = criterion(model(batch_X), batch_y)
                    loss.backward()
                    optimizer.step()

                model.eval()
                with torch.no_grad():
                    es_loss = criterion(model(X_es_t), y_es_t).item()
                    if es_loss < best_fold_loss:
                        best_fold_loss, epochs_no_improve = es_loss, 0
                        best_model_state = model.state_dict().copy()
                    else:
                        epochs_no_improve += 1
                if epochs_no_improve >= patience: break

            model.load_state_dict(best_model_state)
            model.eval()
            with torch.no_grad():
                test_outputs = model(X_test_t)
                test_preds = torch.argmax(test_outputs, dim=1)
                test_acc = (test_preds == y_test_t).sum().item() / len(y_test_t)
                accuracies.append(test_acc)

        current_mean_accuracy = np.mean(accuracies)
        if current_mean_accuracy > max_accuracy:
            max_accuracy, best_params = current_mean_accuracy, (hiddenlayer1, hiddenlayer2)


    # final training on all samples 
    scaler = StandardScaler()
    X_train_full_scaled = scaler.fit_transform(X_train_full)
    X_val_final_scaled = scaler.transform(X_val_final)

    X_train_full_t = torch.FloatTensor(X_train_full_scaled).to(device)
    X_val_final_t = torch.FloatTensor(X_val_final_scaled).to(device)
    y_train_full_t = torch.LongTensor(y_train_full).to(device)

    train_loader = DataLoader(TensorDataset(X_train_full_t, y_train_full_t), batch_size=32, shuffle=True)
    model = ApneaNet(input_size, best_params[0], best_params[1], 3).to(device)
    criterion = nn.CrossEntropyLoss() 
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(100):
        model.train() 
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(batch_X), batch_y)
            loss.backward()
            optimizer.step()

    # valudation 
    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val_final_t)
        val_preds = torch.argmax(val_outputs, dim=1)
        acc, sen, spe, cm = calculate_metrics(y_val_final, val_preds.cpu().numpy()) 

    results = {
        "filepath": filepath,
        "identifier": identifier,
        "model": "MLP multiclass",
        "classes": classes,
        "params": best_params,
        "test_acc": float(max_accuracy),
        "val_acc": float(acc),
        "val_sen": float(sen),
        "val_spe": float(spe),
        "cm" : cm.tolist()
    }
    
    with open(filepath, "a") as f:
        json.dump(results, f)
        f.write('\n')

In [ ]:
# ALL features binary

path_binary_dataset = r"C:\Users\timje\Documents\Studium\bachelor\code_neu\final_datasets\FINAL_dataset_A25_H25_N50.parquet"

DATASET = pd.read_parquet(path_binary_dataset)
DATASET["Apnea"] = DATASET["Apnea"].replace({"Hypopnea": "Apnea"})

dataset_SpO2_ECG, dataset_SpO2, dataset_ECG = clean_dataset2(DATASET)

path = "FINAL_all_binary_results_ALLfeatures.jsonl"

grid_search_RF(dataset=dataset_SpO2_ECG, identifier=f"binary_Spo2+ECG_groupedKFold_allfeatures", classes=2, filepath=f"grid_results\\{path}")
print("done with 1")
grid_search_RF(dataset=dataset_SpO2, identifier=f"binary_Spo2_groupedKFold_allfeatures", classes=2, filepath=f"grid_results\\{path}")
print("done with 2")
grid_search_RF(dataset=dataset_ECG, identifier=f"binary_ECG_groupedKFold_allfeatures", classes=2, filepath=f"grid_results\\{path}")
print("done with A")

grid_search_kNN(dataset=dataset_SpO2_ECG, identifier=f"binary_Spo2+ECG_groupedKFold_allfeatures", classes=2, filepath=f"grid_results\\{path}")
print("done with 1")
grid_search_kNN(dataset=dataset_SpO2, identifier=f"binary_Spo2_groupedKFold_allfeatures", classes=2, filepath=f"grid_results\\{path}")
print("done with 2")
grid_search_kNN(dataset=dataset_ECG, identifier=f"binary_ECG_groupedKFold_allfeatures", classes=2, filepath=f"grid_results\\{path}")
print("done with B")

grid_search_binary_MLP(dataset=dataset_SpO2_ECG, identifier=f"binary_Spo2+ECG_groupedKFold_allfeatures", classes=2, filepath=f"grid_results\\{path}")
print("done with 1")
grid_search_binary_MLP(dataset=dataset_SpO2, identifier=f"binary_Spo2_groupedKFold_allfeatures", classes=2, filepath=f"grid_results\\{path}")
print("done with 2")
grid_search_binary_MLP(dataset=dataset_ECG, identifier=f"binary_ECG_groupedKFold_allfeatures", classes=2, filepath=f"grid_results\\{path}")
print("done with C")

In [ ]:
# binary classification, mRMR 20%

path_binary_dataset = r"C:\Users\timje\Documents\Studium\bachelor\code_neu\final_datasets\FINAL_dataset_A25_H25_N50.parquet"

DATASET = pd.read_parquet(path_binary_dataset)
DATASET["Apnea"] = DATASET["Apnea"].replace({"Hypopnea": "Apnea"})

dataset_reduced_SpO2_ECG, dataset_reduced_SpO2, dataset_reduced_ECG = clean_dataset(DATASET)

# apply MRMR to further reduce amount of features 
dataset_SpO2_ECG =apply_MRMR_framework(dataset_reduced_SpO2_ECG)
dataset_SpO2 = apply_MRMR_framework(dataset_reduced_SpO2)
dataset_ECG = apply_MRMR_framework(dataset_reduced_ECG)

path = "FINAL_all_binary_results_MRMR20.jsonl"

grid_search_RF(dataset=dataset_SpO2_ECG, identifier=f"binary_Spo2+ECG_groupedKFold_MRMR20", classes=2, filepath=f"grid_results\\{path}")
print("done with 1")
grid_search_RF(dataset=dataset_SpO2, identifier=f"binary_Spo2_groupedKFold_MRMR20", classes=2, filepath=f"grid_results\\{path}")
print("done with 2")
grid_search_RF(dataset=dataset_ECG, identifier=f"binary_ECG_groupedKFold_MRMR20", classes=2, filepath=f"grid_results\\{path}")
print("done with A")


grid_search_kNN(dataset=dataset_SpO2_ECG, identifier=f"binary_Spo2+ECG_groupedKFold_MRMR20", classes=2, filepath=f"grid_results\\{path}")
print("done with 1")
grid_search_kNN(dataset=dataset_SpO2, identifier=f"binary_Spo2_groupedKFold_MRMR20", classes=2, filepath=f"grid_results\\{path}")
print("done with 2")
grid_search_kNN(dataset=dataset_ECG, identifier=f"binary_ECG_groupedKFold_MRMR20", classes=2, filepath=f"grid_results\\{path}")
print("done with B")

grid_search_binary_MLP(dataset=dataset_SpO2_ECG, identifier=f"binary_Spo2+ECG_groupedKFold_MRMR20", classes=2, filepath=f"grid_results\\{path}")
print("done with 1")
grid_search_binary_MLP(dataset=dataset_SpO2, identifier=f"binary_Spo2_groupedKFold_MRMR20", classes=2, filepath=f"grid_results\\{path}")
print("done with 2")
grid_search_binary_MLP(dataset=dataset_ECG, identifier=f"binary_ECG_groupedKFold_MRMR20", classes=2, filepath=f"grid_results\\{path}")
print("done with C")

In [ ]:
# multiclass classification mRMR 20%

path_three_dataset = r"C:\Users\timje\Documents\Studium\bachelor\code_neu\final_datasets\FINAL_dataset_A33_H33_N33.parquet"

DATASET = pd.read_parquet(path_three_dataset)

dataset_reduced_SpO2_ECG, dataset_reduced_SpO2, dataset_reduced_ECG = clean_dataset(DATASET)

dataset_SpO2_ECG =apply_MRMR_framework(dataset_reduced_SpO2_ECG)
dataset_SpO2 = apply_MRMR_framework(dataset_reduced_SpO2)
dataset_ECG = apply_MRMR_framework(dataset_reduced_ECG)

path = "FINAL_all_multiclass_results_MRMR20.jsonl"


grid_search_RF(dataset=dataset_SpO2_ECG, identifier=f"multiclass_Spo2+ECG_groupedKFold_MRMR20", classes=3, filepath=f"grid_results\\{path}")
print("done with 1")
grid_search_RF(dataset=dataset_SpO2, identifier=f"multiclass_Spo2_groupedKFold_MRMR20", classes=3, filepath=f"grid_results\\{path}")
print("done with 2")
grid_search_RF(dataset=dataset_ECG, identifier=f"multiclass_ECG_groupedKFold_MRMR20", classes=3, filepath=f"grid_results\\{path}")
print("done with D")

grid_search_kNN(dataset=dataset_SpO2_ECG, identifier=f"multiclass_Spo2+ECG_groupedKFold_MRMR20", classes=3, filepath=f"grid_results\\{path}")
print("done with 1")
grid_search_kNN(dataset=dataset_SpO2, identifier=f"multiclass_Spo2_groupedKFold_MRMR20", classes=3, filepath=f"grid_results\\{path}")
print("done with 2")
grid_search_kNN(dataset=dataset_ECG, identifier=f"multiclass_ECG_groupedKFold_MRMR20", classes=3, filepath=f"grid_results\\{path}")
print("done with E")

grid_search_MLP_three_classes(dataset=dataset_SpO2_ECG, identifier=f"multiclass_Spo2+ECG_groupedKFold_MRMR20", classes=3, filepath=f"grid_results\\{path}")
print("done with 1")
grid_search_MLP_three_classes(dataset=dataset_SpO2, identifier=f"multiclass_Spo2_groupedKFold_MRMR20", classes=3, filepath=f"grid_results\\{path}")
print("done with 2")
grid_search_MLP_three_classes(dataset=dataset_ECG, identifier=f"multiclass_ECG_groupedKFold_MRMR20", classes=3, filepath=f"grid_results\\{path}")
print("done with F")

In [ ]:
# binary classification, only F-test, no mRMR

path_binary_dataset = r"C:\Users\timje\Documents\Studium\bachelor\code_neu\final_datasets\FINAL_dataset_A25_H25_N50.parquet"

DATASET = pd.read_parquet(path_binary_dataset)
DATASET["Apnea"] = DATASET["Apnea"].replace({"Hypopnea": "Apnea"})

dataset_SpO2_ECG, dataset_SpO2, dataset_ECG = clean_dataset(DATASET)


path = "FINAL_all_binary_results_onlyFtest.jsonl"

grid_search_RF(dataset=dataset_SpO2_ECG, identifier=f"binary_Spo2+ECG_groupedKFold", classes=2, filepath=f"grid_results\\{path}")
print("done with 1")
grid_search_RF(dataset=dataset_SpO2, identifier=f"binary_Spo2_groupedKFold", classes=2, filepath=f"grid_results\\{path}")
print("done with 2")
grid_search_RF(dataset=dataset_ECG, identifier=f"binary_ECG_groupedKFold", classes=2, filepath=f"grid_results\\{path}")
print("done with A")


grid_search_kNN(dataset=dataset_SpO2_ECG, identifier=f"binary_Spo2+ECG_groupedKFold", classes=2, filepath=f"grid_results\\{path}")
print("done with 1")
grid_search_kNN(dataset=dataset_SpO2, identifier=f"binary_Spo2_groupedKFold", classes=2, filepath=f"grid_results\\{path}")
print("done with 2")
grid_search_kNN(dataset=dataset_ECG, identifier=f"binary_ECG_groupedKFold", classes=2, filepath=f"grid_results\\{path}")
print("done with B")

grid_search_binary_MLP(dataset=dataset_SpO2_ECG, identifier=f"binary_Spo2+ECG_groupedKFold", classes=2, filepath=f"grid_results\\{path}")
print("done with 1")
grid_search_binary_MLP(dataset=dataset_SpO2, identifier=f"binary_Spo2_groupedKFold", classes=2, filepath=f"grid_results\\{path}")
print("done with 2")
grid_search_binary_MLP(dataset=dataset_ECG, identifier=f"binary_ECG_groupedKFold", classes=2, filepath=f"grid_results\\{path}")

In [ ]:
# multiclass classification, only F-test, no mRMR

path_three_dataset = r"C:\Users\timje\Documents\Studium\bachelor\code_neu\final_datasets\FINAL_dataset_A33_H33_N33.parquet"

DATASET = pd.read_parquet(path_three_dataset)

dataset_SpO2_ECG, dataset_SpO2, dataset_ECG = clean_dataset(DATASET)

path = "FINAL_all_multiclass_results_onlyFtest.jsonl"


grid_search_RF(dataset=dataset_SpO2_ECG, identifier=f"multiclass_Spo2+ECG_groupedKFold", classes=3, filepath=f"grid_results\\{path}")
print("done with 1")
grid_search_RF(dataset=dataset_SpO2, identifier=f"multiclass_Spo2_groupedKFold", classes=3, filepath=f"grid_results\\{path}")
print("done with 2")
grid_search_RF(dataset=dataset_ECG, identifier=f"multiclass_ECG_groupedKFold", classes=3, filepath=f"grid_results\\{path}")
print("done with D")

grid_search_kNN(dataset=dataset_SpO2_ECG, identifier=f"multiclass_Spo2+ECG_groupedKFold", classes=3, filepath=f"grid_results\\{path}")
print("done with 1")
grid_search_kNN(dataset=dataset_SpO2, identifier=f"multiclass_Spo2_groupedKFold", classes=3, filepath=f"grid_results\\{path}")
print("done with 2")
grid_search_kNN(dataset=dataset_ECG, identifier=f"multiclass_ECG_groupedKFold", classes=3, filepath=f"grid_results\\{path}")
print("done with E")

grid_search_MLP_three_classes(dataset=dataset_SpO2_ECG, identifier=f"multiclass_Spo2+ECG_groupedKFold", classes=3, filepath=f"grid_results\\{path}")
print("done with 1")
grid_search_MLP_three_classes(dataset=dataset_SpO2, identifier=f"multiclass_Spo2_groupedKFold", classes=3, filepath=f"grid_results\\{path}")
print("done with 2")
grid_search_MLP_three_classes(dataset=dataset_ECG, identifier=f"multiclass_ECG_groupedKFold", classes=3, filepath=f"grid_results\\{path}")

In [ ]:
# binary classification MRMR50

path_binary_dataset = r"C:\Users\timje\Documents\Studium\bachelor\code_neu\final_datasets\FINAL_dataset_A25_H25_N50.parquet"

DATASET = pd.read_parquet(path_binary_dataset)
DATASET["Apnea"] = DATASET["Apnea"].replace({"Hypopnea": "Apnea"})

dataset_reduced_SpO2_ECG, dataset_reduced_SpO2, dataset_reduced_ECG = clean_dataset(DATASET)

# apply MRMR to further reduce amount of features 
dataset_SpO2_ECG =apply_MRMR_framework_percent(dataset_reduced_SpO2_ECG, keep_percent=0.5)
dataset_SpO2 = apply_MRMR_framework_percent(dataset_reduced_SpO2, keep_percent=0.5)
dataset_ECG = apply_MRMR_framework_percent(dataset_reduced_ECG, keep_percent=0.5)

path = "FINAL_all_binary_results_MRMR50.jsonl"


grid_search_RF(dataset=dataset_SpO2_ECG, identifier=f"binary_Spo2+ECG_groupedKFold_MRMR50", classes=2, filepath=f"grid_results\\{path}")
print("done with 1")
grid_search_RF(dataset=dataset_SpO2, identifier=f"binary_Spo2_groupedKFold_MRMR50", classes=2, filepath=f"grid_results\\{path}")
print("done with 2")
grid_search_RF(dataset=dataset_ECG, identifier=f"binary_ECG_groupedKFold_MRMR50", classes=2, filepath=f"grid_results\\{path}")
print("done with A")

grid_search_kNN(dataset=dataset_SpO2_ECG, identifier=f"binary_Spo2+ECG_groupedKFold_MRMR50", classes=2, filepath=f"grid_results\\{path}")
print("done with 1")
grid_search_kNN(dataset=dataset_SpO2, identifier=f"binary_Spo2_groupedKFold_MRMR50", classes=2, filepath=f"grid_results\\{path}")
print("done with 2")
grid_search_kNN(dataset=dataset_ECG, identifier=f"binary_ECG_groupedKFold_MRMR50", classes=2, filepath=f"grid_results\\{path}")
print("done with B")

grid_search_binary_MLP(dataset=dataset_SpO2_ECG, identifier=f"binary_Spo2+ECG_groupedKFold_MRMR50", classes=2, filepath=f"grid_results\\{path}")
print("done with 1")
grid_search_binary_MLP(dataset=dataset_SpO2, identifier=f"binary_Spo2_groupedKFold_MRMR50", classes=2, filepath=f"grid_results\\{path}")
print("done with 2")
grid_search_binary_MLP(dataset=dataset_ECG, identifier=f"binary_ECG_groupedKFold_MRMR50", classes=2, filepath=f"grid_results\\{path}")
print("done with C")

In [ ]:
# multiclass classification MRMR50

path_three_dataset = r"C:\Users\timje\Documents\Studium\bachelor\code_neu\final_datasets\FINAL_dataset_A33_H33_N33.parquet"

DATASET = pd.read_parquet(path_three_dataset)

dataset_reduced_SpO2_ECG, dataset_reduced_SpO2, dataset_reduced_ECG = clean_dataset(DATASET)

dataset_SpO2_ECG =apply_MRMR_framework_percent(dataset_reduced_SpO2_ECG, keep_percent=0.5)
dataset_SpO2 = apply_MRMR_framework_percent(dataset_reduced_SpO2, keep_percent=0.5)
dataset_ECG = apply_MRMR_framework_percent(dataset_reduced_ECG, keep_percent=0.5)

path = "FINAL_all_multiclass_results_MRMR50.jsonl"


grid_search_RF(dataset=dataset_SpO2_ECG, identifier=f"multiclass_Spo2+ECG_groupedKFold_MRMR50", classes=3, filepath=f"grid_results\\{path}")
print("done with 1")
grid_search_RF(dataset=dataset_SpO2, identifier=f"multiclass_Spo2_groupedKFold_MRMR50", classes=3, filepath=f"grid_results\\{path}")
print("done with 2")
grid_search_RF(dataset=dataset_ECG, identifier=f"multiclass_ECG_groupedKFold_MRMR50", classes=3, filepath=f"grid_results\\{path}")
print("done with D")

grid_search_kNN(dataset=dataset_SpO2_ECG, identifier=f"multiclass_Spo2+ECG_groupedKFold_MRMR50", classes=3, filepath=f"grid_results\\{path}")
print("done with 1")
grid_search_kNN(dataset=dataset_SpO2, identifier=f"multiclass_Spo2_groupedKFold_MRMR50", classes=3, filepath=f"grid_results\\{path}")
print("done with 2")
grid_search_kNN(dataset=dataset_ECG, identifier=f"multiclass_ECG_groupedKFold_MRMR50", classes=3, filepath=f"grid_results\\{path}")
print("done with E")

grid_search_MLP_three_classes(dataset=dataset_SpO2_ECG, identifier=f"multiclass_Spo2+ECG_groupedKFold_MRMR50", classes=3, filepath=f"grid_results\\{path}")
print("done with 1")
grid_search_MLP_three_classes(dataset=dataset_SpO2, identifier=f"multiclass_Spo2_groupedKFold_MRMR50", classes=3, filepath=f"grid_results\\{path}")
print("done with 2")
grid_search_MLP_three_classes(dataset=dataset_ECG, identifier=f"multiclass_ECG_groupedKFold_MRMR50", classes=3, filepath=f"grid_results\\{path}")
print("done with F")

In [ ]:
# binary classification, MRMR50 and only 50% of the most important features according to RF


bin_spo2_ecg = ["ODI21", "CTM25", "ODI23", "ODI33", "odi2", "scc2", "LZCom", "rr_psd_fft32_8", "rr_psd_fft32_10", "CorreC", "NN20", "scc5", "nep", "rr_psd_fft32_29", "tsa95", "tsa90", "tsa85", "edr_DWT_var_VLF", "tsa70", "edr_mean", "edr_psd_fft32_0"]
bin_spo2 = ["CTM25", "ODI23", "odi2", "LZCom", "tsa95", "CorreC", "tsa90", "tsa85", "tsa70"]
bin_ecg = ["scc2", "scc1", "rr_psd_fft32_8", "scc5", "rr_psd_fft32_29", "rr_psd_fft32_10", "rr_DWT_var_1", "nep", "NN20", "edr_psd_fft32_28", "edr_mean", "edr_psd_fft32_0"]
to_keep = ["Apnea","Subject","Age","Sex","AHI"]

path_binary_dataset = r"C:\Users\timje\Documents\Studium\bachelor\code_neu\final_datasets\FINAL_dataset_A25_H25_N50.parquet"

DATASET = pd.read_parquet(path_binary_dataset)
DATASET["Apnea"] = DATASET["Apnea"].replace({"Hypopnea": "Apnea"})

dataset_reduced_SpO2_ECG, dataset_reduced_SpO2, dataset_reduced_ECG = clean_dataset(DATASET)

dataset_SpO2_ECG = dataset_reduced_SpO2_ECG[to_keep + bin_spo2_ecg[:int(len(bin_spo2_ecg)/2)]]
dataset_SpO2 = dataset_reduced_SpO2[to_keep + bin_spo2[:int(len(bin_spo2)/2)]]
dataset_ECG = dataset_reduced_ECG[to_keep + bin_ecg[:int(len(bin_ecg)/2)]]

path = "FINAL_all_binary_results_MRMR50_RF50.jsonl"


grid_search_RF(dataset=dataset_SpO2_ECG, identifier=f"binary_Spo2+ECG_groupedKFold_MRMR50_RF50", classes=2, filepath=f"grid_results\\{path}")
print("done with 1")
grid_search_RF(dataset=dataset_SpO2, identifier=f"binary_Spo2_groupedKFold_MRMR50_RF50", classes=2, filepath=f"grid_results\\{path}")
print("done with 2")
grid_search_RF(dataset=dataset_ECG, identifier=f"binary_ECG_groupedKFold_MRMR50_RF50", classes=2, filepath=f"grid_results\\{path}")
print("done with A")

grid_search_kNN(dataset=dataset_SpO2_ECG, identifier=f"binary_Spo2+ECG_groupedKFold_MRMR50_RF50", classes=2, filepath=f"grid_results\\{path}")
print("done with 1")
grid_search_kNN(dataset=dataset_SpO2, identifier=f"binary_Spo2_groupedKFold_MRMR50_RF50", classes=2, filepath=f"grid_results\\{path}")
print("done with 2")
grid_search_kNN(dataset=dataset_ECG, identifier=f"binary_ECG_groupedKFold_MRMR50_RF50", classes=2, filepath=f"grid_results\\{path}")
print("done with B")

grid_search_binary_MLP(dataset=dataset_SpO2_ECG, identifier=f"binary_Spo2+ECG_groupedKFold_MRMR50_RF50", classes=2, filepath=f"grid_results\\{path}")
print("done with 1")
grid_search_binary_MLP(dataset=dataset_SpO2, identifier=f"binary_Spo2_groupedKFold_MRMR50_RF50", classes=2, filepath=f"grid_results\\{path}")
print("done with 2")
grid_search_binary_MLP(dataset=dataset_ECG, identifier=f"binary_ECG_groupedKFold_MRMR50_RF50", classes=2, filepath=f"grid_results\\{path}")
print("done with C")


In [ ]:
# multiclass classifcation, MRMR50 and only 50% of the most important features according to RF

multi_spo2_ecg = ["CTM25", "ODI31", "ODI21", "ODI35", "odi2", "ODI25", "AbsSlope", "CTM100", "rr_psd_fft32_4", "scc2", "rr_psd_fft32_7", "rr_sd", "LZCom", "tsa95", "scc5", "nep", "tsa85", "tsa70", "edr_mean", "edr_psd_fft32_0"]
multi_spo2 = ["CTM25", "odi2", "CorreC", "ODI25", "ODI35", "tsa95", "LZCom", "tsa85", "tsa70"]
multi_ecg = ["scc2", "rr_psd_fft32_4", "scc1", "rr_psd_fft32_7", "rr_sd", "scc5", "rr_mean", "edr_psd_fft32_29", "edr_psd_fft32_2", "edr_mean", "edr_psd_fft32_0"]

to_keep = ["Apnea","Subject","Age","Sex","AHI"]

path_three_dataset = r"C:\Users\timje\Documents\Studium\bachelor\code_neu\final_datasets\FINAL_dataset_A33_H33_N33.parquet"

DATASET = pd.read_parquet(path_three_dataset)

dataset_reduced_SpO2_ECG, dataset_reduced_SpO2, dataset_reduced_ECG = clean_dataset(DATASET)

dataset_SpO2_ECG = dataset_reduced_SpO2_ECG[to_keep + multi_spo2_ecg[:int(len(multi_spo2_ecg)/2)]]
dataset_SpO2 = dataset_reduced_SpO2[to_keep + multi_spo2[:int(len(multi_spo2)/2)]]
dataset_ECG = dataset_reduced_ECG[to_keep + multi_ecg[:int(len(multi_ecg)/2)]]

path = "FINAL_all_multiclass_results_MRMR50_RF50.jsonl"


grid_search_RF(dataset=dataset_SpO2_ECG, identifier=f"multiclass_Spo2+ECG_groupedKFold_MRMR50_RF50", classes=3, filepath=f"grid_results\\{path}")
print("done with 1")
grid_search_RF(dataset=dataset_SpO2, identifier=f"multiclass_Spo2_groupedKFold_MRMR50_RF50", classes=3, filepath=f"grid_results\\{path}")
print("done with 2")
grid_search_RF(dataset=dataset_ECG, identifier=f"multiclass_ECG_groupedKFold_MRMR50_RF50", classes=3, filepath=f"grid_results\\{path}")
print("done with D")

grid_search_kNN(dataset=dataset_SpO2_ECG, identifier=f"multiclass_Spo2+ECG_groupedKFold_MRMR50_RF50", classes=3, filepath=f"grid_results\\{path}")
print("done with 1")
grid_search_kNN(dataset=dataset_SpO2, identifier=f"multiclass_Spo2_groupedKFold_MRMR50_RF50", classes=3, filepath=f"grid_results\\{path}")
print("done with 2")
grid_search_kNN(dataset=dataset_ECG, identifier=f"multiclass_ECG_groupedKFold_MRMR50_RF50", classes=3, filepath=f"grid_results\\{path}")
print("done with E")

grid_search_MLP_three_classes(dataset=dataset_SpO2_ECG, identifier=f"multiclass_Spo2+ECG_groupedKFold_MRMR50_RF50", classes=3, filepath=f"grid_results\\{path}")
print("done with 1")
grid_search_MLP_three_classes(dataset=dataset_SpO2, identifier=f"multiclass_Spo2_groupedKFold_MRMR50_RF50", classes=3, filepath=f"grid_results\\{path}")
print("done with 2")
grid_search_MLP_three_classes(dataset=dataset_ECG, identifier=f"multiclass_ECG_groupedKFold_MRMR50_RF50", classes=3, filepath=f"grid_results\\{path}")
print("done with F")